# Science Discovery Agent - run it from your phone

No installation needed. Everything runs in this browser tab, on Google's free servers.

**How to use (3 steps):**

1. Tap the **Runtime** menu at the top, then **Run all**. (Or tap the play button on each cell, top to bottom.)
2. When a cell asks for your **API key**, paste it and press Enter. Get a free/cheap key from Sarvam AI, Groq, or OpenAI - see the notes in the cell below.
3. In the **last cell**, change the research question to whatever you want, and run that cell again.

You can do all of this on a phone. Nothing is saved to your phone.

## Step 1 - install the one library we need

In [ ]:
!pip install openai -q


## Step 2 - add your API key

Get a key from one of these (all work, pick one):

- **Groq** - has a free tier, very fast. Go to console.groq.com
- **Sarvam AI** - Indian, low cost. Go to dashboard.sarvam.ai
- **OpenAI** - paid. Go to platform.openai.com

The key is hidden as you type it, and it only lives in this session - it is never saved to your phone.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Paste your API key, then press Enter: ").strip()

# If you are NOT using OpenAI, also set your provider's endpoint and model.
# Uncomment the lines below and fill them in:

# os.environ["OPENAI_BASE_URL"] = "https://your-provider-endpoint/v1"
# os.environ["MODEL"] = "your-model-name"

print("Key set:", "yes" if os.environ.get("OPENAI_API_KEY") else "NO - go back and paste it")


## Step 3 - load the agent (just run this cell, do not edit it)

This is the whole system: the Scout -> Generator -> Critic -> Judge loop, plus a
real multi-source literature search (arXiv, OpenAlex, Crossref, Semantic Scholar,
Europe PMC - no extra keys needed).

In [ ]:
#!/usr/bin/env python3
"""
science_discovery_agent.py
==========================
A runnable scaffold for a SCIENTIFIC DISCOVERY agent.

It runs a multi-agent loop:

    Scout  ->  Generator  ->  Critic  ->  Judge
    (what's     (new           (is it      (rank &
     known /     hypotheses)    novel &     report)
     gaps)                      sound?)

The core idea: a single LLM call does NOT invent something new. Discovery
comes from a loop that (a) grounds the model in what already exists,
(b) generates many candidate hypotheses, (c) attacks those hypotheses
adversarially, and (d) ranks the survivors. THAT loop is the thing you
iterate on and plug real scientific tools into.

--------------------------------------------------------------------------------
HOW TO RUN
--------------------------------------------------------------------------------
1)  pip install openai
2)  export OPENAI_API_KEY="..."        # your key (never hard-code it)
    export OPENAI_BASE_URL="..."       # optional: any OpenAI-compatible endpoint
    export MODEL="gpt-4o-mini"         # optional, defaults below
    export CONTACT_EMAIL="you@x.com"   # optional: polite-pool courtesy
3)  python science_discovery_agent.py "Your research question here"

Works with any OpenAI-compatible API (Sarvam, Together, Groq, a local server)
by pointing OPENAI_BASE_URL at it.

--------------------------------------------------------------------------------
LITERATURE SEARCH
--------------------------------------------------------------------------------
`literature_search()` queries FIVE open scientific databases in parallel -- no
API key needed for any of them -- and merges the results:

    arXiv            physics, maths, CS, quantitative biology, preprints
    OpenAlex         all disciplines (successor to MS Academic Graph)
    Crossref         all disciplines, DOI registry
    Semantic Scholar all disciplines
    Europe PMC       life sciences / biomedicine

Any source that is down, blocked, or rate-limited is skipped silently, so the
agent keeps working with whatever it can reach. This is what gives the Critic
a real basis for judging novelty -- without it, the agent has no idea what
already exists and will "rediscover" known facts.
"""

from __future__ import annotations

import json
import os
import re
import sys
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict

# --------------------------------------------------------------------------- #
# LLM client (lazy import so the file compiles without the package installed)
# --------------------------------------------------------------------------- #

_MODEL = os.environ.get("MODEL", "gpt-4o-mini")


def _client():
    try:
        from openai import OpenAI  # type: ignore
    except ImportError:
        sys.exit("Missing dependency. Run:  pip install openai")
    if not os.environ.get("OPENAI_API_KEY"):
        sys.exit("Set your key first:  export OPENAI_API_KEY='...'")
    base = os.environ.get("OPENAI_BASE_URL")
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=base) if base \
        else OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def call_llm(system: str, user: str, *, json_mode: bool = False) -> str:
    """One LLM round-trip. Returns raw text."""
    kwargs = {"model": _MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": user}],
              "temperature": 0.7}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = _client().chat.completions.create(**kwargs)
    return resp.choices[0].message.content or ""


def _parse_json(text: str):
    """Best-effort JSON extraction from an LLM reply."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return None


# --------------------------------------------------------------------------- #
# TOOLS  --  real, multi-source literature search (no API keys required)
# --------------------------------------------------------------------------- #

CONTACT = os.environ.get("CONTACT_EMAIL", "you@example.com")


def _http_get(url: str, timeout: int = 20) -> str:
    req = urllib.request.Request(url, headers={
        "User-Agent": f"science-discovery-agent/0.1 (mailto:{CONTACT})",
        "Accept": "*/*",
    })
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace")


def _reconstruct_abstract(inv) -> str:
    """OpenAlex stores abstracts as an inverted index; rebuild plain text."""
    if not inv:
        return ""
    pos = {}
    for word, idxs in inv.items():
        for i in idxs:
            pos[i] = word
    return " ".join(pos[i] for i in sorted(pos))


def _src_arxiv(query, k):
    # NOTE: do NOT wrap the query in quotes -- arXiv treats a quoted string as an
    # exact phrase and returns nothing for a full research question. Bare terms
    # are AND-ed, which is what we want.
    q = urllib.parse.quote(f"all:{query}")
    url = (f"http://export.arxiv.org/api/query?search_query={q}"
           f"&max_results={k}&sortBy=relevance")
    root = ET.fromstring(_http_get(url))
    ns = {"a": "http://www.w3.org/2005/Atom"}
    out = []
    for e in root.findall("a:entry", ns):
        out.append({
            "title": " ".join(e.find("a:title", ns).text.split()),
            "year": e.find("a:published", ns).text[:4],
            "snippet": " ".join(e.find("a:summary", ns).text.split())[:300],
            "source": "arXiv",
        })
    return out


def _src_openalex(query, k):
    url = (f"https://api.openalex.org/works?search={urllib.parse.quote(query)}"
           f"&per_page={k}&mailto={CONTACT}")
    d = json.loads(_http_get(url))
    out = []
    for w in d.get("results", []):
        out.append({
            "title": w.get("title") or "(untitled)",
            "year": str(w.get("publication_year") or "—"),
            "snippet": _reconstruct_abstract(w.get("abstract_inverted_index"))[:300],
            "source": "OpenAlex",
        })
    return out


def _src_crossref(query, k):
    url = (f"https://api.crossref.org/works?query={urllib.parse.quote(query)}"
           f"&rows={k}&mailto={CONTACT}&select=title,issued,abstract")
    d = json.loads(_http_get(url))
    out = []
    for it in d.get("message", {}).get("items", []):
        title = (it.get("title") or ["(untitled)"])[0]
        dp = it.get("issued", {}).get("date-parts", [[None]])
        year = dp[0][0] if dp and dp[0] else None
        out.append({
            "title": title,
            "year": str(year or "—"),
            "snippet": re.sub("<[^>]+>", "", it.get("abstract") or "")[:300],
            "source": "Crossref",
        })
    return out


def _src_semanticscholar(query, k):
    url = ("https://api.semanticscholar.org/graph/v1/paper/search?"
           f"query={urllib.parse.quote(query)}&limit={k}"
           "&fields=title,year,abstract")
    d = json.loads(_http_get(url))
    return [{
        "title": p.get("title") or "(untitled)",
        "year": str(p.get("year") or "—"),
        "snippet": (p.get("abstract") or "")[:300],
        "source": "SemanticScholar",
    } for p in d.get("data", [])]


def _src_europepmc(query, k):
    url = ("https://www.ebi.ac.uk/europepmc/webservices/rest/search?"
           f"query={urllib.parse.quote(query)}&format=json&pageSize={k}")
    d = json.loads(_http_get(url))
    out = []
    for r in d.get("resultList", {}).get("result", []):
        out.append({
            "title": r.get("title") or "(untitled)",
            "year": str(r.get("pubYear") or "—"),
            "snippet": (r.get("abstractText") or "")[:300],
            "source": "EuropePMC",
        })
    return out


# Register sources here. Comment one out if you don't want it queried.
_SOURCES = [
    ("arXiv", _src_arxiv),
    ("OpenAlex", _src_openalex),
    ("Crossref", _src_crossref),
    ("SemanticScholar", _src_semanticscholar),
    ("EuropePMC", _src_europepmc),
]


def literature_search(query: str, k: int = 6) -> list[dict]:
    """
    Query several open scientific databases in parallel and merge the hits.
    Returns a list of {title, year, snippet, source}. Degrades gracefully:
    a source that is down, blocked, or rate-limited is simply skipped.
    """
    results, seen, errors = [], set(), []
    per_source = max(2, k // 2)

    with ThreadPoolExecutor(max_workers=len(_SOURCES)) as ex:
        futs = {ex.submit(fn, query, per_source): name for name, fn in _SOURCES}
        for fut in as_completed(futs):
            name = futs[fut]
            try:
                for h in fut.result():
                    key = (h.get("title") or "").strip().lower()[:80]
                    if key and key not in seen:
                        seen.add(key)
                        results.append(h)
            except Exception as e:  # source down / blocked / rate-limited
                errors.append(f"{name}: {type(e).__name__}")

    if not results:
        results = [{
            "title": "[no literature source reachable]",
            "year": "—",
            "snippet": "Sources failed: " + ("; ".join(errors) or "unknown"),
            "source": "none",
        }]
    return results[:k]


# --------------------------------------------------------------------------- #
# DATA MODEL
# --------------------------------------------------------------------------- #

@dataclass
class Hypothesis:
    statement: str
    rationale: str = ""
    prediction: str = ""          # what experiment/observation would test it
    novelty: int = 0              # 1-5, from the Critic
    soundness: int = 0            # 1-5, from the Critic
    feasibility: int = 0          # 1-5, from the Critic
    critique: str = ""
    score: float = 0.0

    def compute_score(self) -> float:
        # tune these weights for your field
        self.score = round(
            0.45 * self.novelty + 0.35 * self.soundness + 0.20 * self.feasibility, 3)
        return self.score


@dataclass
class DiscoveryState:
    question: str
    rounds: int = 2
    known: str = ""
    gaps: list[str] = field(default_factory=list)
    hypotheses: list[Hypothesis] = field(default_factory=list)
    report: str = ""


def _fmt_hits(hits: list[dict]) -> str:
    return "\n".join(
        f"- {h['title']} ({h['year']}) [{h.get('source', '?')}]: {h['snippet'][:200]}"
        for h in hits)


# --------------------------------------------------------------------------- #
# AGENTS
# --------------------------------------------------------------------------- #

def scout(state: DiscoveryState) -> None:
    """Ground the model: what is known, and where are the gaps?"""
    hits = literature_search(state.question)
    lit = _fmt_hits(hits)

    raw = call_llm(
        "You are a careful research librarian. Be concise and factual. "
        "Never invent citations.",
        f"""Research question: {state.question}

Literature retrieved (may be incomplete):
{lit}

In JSON with keys "known" (a short paragraph) and "gaps" (a list of 3-5
specific, open questions NOT answered by the literature above), describe the
current state of knowledge and the most promising gaps.""",
        json_mode=True,
    )
    data = _parse_json(raw) or {}
    state.known = data.get("known", "")
    state.gaps = data.get("gaps", []) or []


def generate(state: DiscoveryState, n: int = 6) -> None:
    """Propose many candidate hypotheses aimed at the gaps."""
    raw = call_llm(
        "You are a creative but rigorous scientist. Propose testable hypotheses. "
        "Each must be falsifiable and make a concrete prediction.",
        f"""Research question: {state.question}

What is known: {state.known}
Open gaps: {json.dumps(state.gaps)}

Propose {n} candidate hypotheses that could close one of these gaps.
Return JSON: {{"hypotheses": [{{"statement": ..., "rationale": ...,
"prediction": ...}}]}}""",
        json_mode=True,
    )
    data = _parse_json(raw) or {}
    for h in data.get("hypotheses", [])[:n]:
        if isinstance(h, dict) and h.get("statement"):
            state.hypotheses.append(Hypothesis(
                statement=h.get("statement", ""),
                rationale=h.get("rationale", ""),
                prediction=h.get("prediction", ""),
            ))


def critique(state: DiscoveryState) -> None:
    """Adversarially score each hypothesis: novelty, soundness, feasibility."""
    for h in state.hypotheses:
        hits = literature_search(h.statement)
        lit = _fmt_hits(hits[:5])
        raw = call_llm(
            "You are a harsh, sceptical peer reviewer. Your job is to find why a "
            "hypothesis is wrong, already known, or untestable. Score honestly; "
            "do not be generous.",
            f"""Hypothesis: {h.statement}
Rationale: {h.rationale}
Predicted test: {h.prediction}

Closest prior work found:
{lit}

Score each 1-5 (5 = best):
  novelty   = is this genuinely new vs the prior work above?
  soundness = does it follow from established science without a logical leap?
  feasibility = can it be tested with realistic resources?
Return JSON: {{"novelty": n, "soundness": n, "feasibility": n, "critique": "..."}}""",
            json_mode=True,
        )
        d = _parse_json(raw) or {}
        h.novelty = int(d.get("novelty", 0) or 0)
        h.soundness = int(d.get("soundness", 0) or 0)
        h.feasibility = int(d.get("feasibility", 0) or 0)
        h.critique = d.get("critique", "")
        h.compute_score()


def judge(state: DiscoveryState, top_k: int = 3) -> None:
    """Rank survivors and write a short report."""
    ranked = sorted(state.hypotheses, key=lambda x: x.score, reverse=True)[:top_k]
    state.hypotheses = ranked
    bullets = "\n".join(
        f"{i+1}. (score {h.score}) {h.statement}\n   test: {h.prediction}\n"
        f"   critic: {h.critique}"
        for i, h in enumerate(ranked)
    )
    state.report = call_llm(
        "You are a scientific writer. Be clear about uncertainty. Do not "
        "overstate novelty.",
        f"""Research question: {state.question}

Top surviving hypotheses after adversarial review:
{bullets}

Write a short report (200-300 words): what the question is, which hypotheses
survived and why, and what the single most valuable next experiment would be.""",
    )


# --------------------------------------------------------------------------- #
# ORCHESTRATION
# --------------------------------------------------------------------------- #

def run(question: str, rounds: int = 2) -> DiscoveryState:
    state = DiscoveryState(question=question, rounds=rounds)
    print(f"\n=== Research question: {question} ===\n")

    scout(state)
    print("GAPS FOUND:")
    for g in state.gaps:
        print(f"  - {g}")

    for r in range(rounds):
        print(f"\n--- round {r + 1}: generating + critiquing ---")
        generate(state, n=6)
        critique(state)
        state.hypotheses.sort(key=lambda x: x.score, reverse=True)
        state.hypotheses = state.hypotheses[:8]   # survivors seed the next round
        print(f"  {len(state.hypotheses)} hypotheses carried forward")

    judge(state)
    print("\n=== REPORT ===\n")
    print(state.report)
    return state


def main() -> None:
    if len(sys.argv) < 2:
        sys.exit('Usage: python science_discovery_agent.py "your research question"')
    question = " ".join(sys.argv[1:])
    state = run(question)

    out = "/scratch/work/discovery_result.json"
    with open(out, "w") as f:
        json.dump({"question": state.question, "gaps": state.gaps,
                   "report": state.report,
                   "hypotheses": [asdict(h) for h in state.hypotheses]},
                  f, indent=2)
    print(f"\nSaved structured result -> {out}")

# (CLI entrypoint removed - in Colab you call run() yourself, see the next cell)



## Step 4 - ask your question

Edit the text in quotes below to your own research question, then run this cell.
It takes a minute or two. You will see the gaps, then each round, then a report.

In [ ]:
import json
from dataclasses import asdict

question = "Why do some RNA-binding proteins form condensates under heat stress?"

state = run(question, rounds=2)

with open("discovery_result.json", "w") as f:
    json.dump({"question": state.question, "gaps": state.gaps, "report": state.report,
               "hypotheses": [asdict(h) for h in state.hypotheses]}, f, indent=2)
print("\nSaved discovery_result.json - open the folder icon on the left to download it.")


## Remember

This gives you **candidate** hypotheses, not discoveries. Real novelty is
confirmed by an experiment, proof, or observation - by a person or a lab. The
agent speeds up the thinking, it does not replace the testing.